# Week 6 Deliverable  Simple Neural Network Architecture

> Build and understand a simple neural network architecture while connecting layers, activation functions, forward propagation, loss calculation, backpropagation, and optimization.

This notebook uses a small synthetic dataset so the focus stays on how a neural network is structured and how it learns.

## 1. Import Required Libraries

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

ModuleNotFoundError: No module named 'tensorflow'

## 2. Create a Very Simple Dataset

The following code creates two interleaving moon shapes with NumPy. No external file is needed. Each point has two input features, and its color shows one of two classes.

In [ ]:
samples_per_class = 200
angles = np.random.uniform(0, np.pi, samples_per_class)
noise = 0.12

moon_0 = np.column_stack((np.cos(angles), np.sin(angles)))
moon_1 = np.column_stack((1 - np.cos(angles), 0.5 - np.sin(angles)))
moon_0 += np.random.normal(0, noise, moon_0.shape)
moon_1 += np.random.normal(0, noise, moon_1.shape)

X = np.vstack((moon_0, moon_1)).astype(np.float32)
y = np.concatenate((np.zeros(samples_per_class), np.ones(samples_per_class))).astype(np.float32)

indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]
split_index = int(0.8 * len(X))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", edgecolor="k", alpha=0.8)
plt.title("Synthetic Binary Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()

## 3. Perceptron: The Basic Neuron

```text
Inputs
  ↓
Weights
  ↓
Weighted Sum + Bias
  ↓
Activation Function
  ↓
Output
```

```text
z = w1x1 + w2x2 + ... + b
output = activation(z)
```

- **Input:** a feature supplied to the neuron.
- **Weight:** controls how strongly an input affects the result.
- **Bias:** an adjustable value that shifts the weighted sum.
- **Weighted sum:** combines the inputs, weights, and bias.
- **Activation function:** transforms the weighted sum.
- **Output:** the value passed to the next layer or used as a prediction.

## 4. Neural Network Layers

```text
Input Layer
    ↓
Hidden Layer
    ↓
Output Layer
```

### Input Layer
Receives the input features. Here, each sample has two features.

### Hidden Layer
Learns useful patterns and combinations from the inputs.

### Output Layer
Produces the final prediction.

A single perceptron contains one neuron and can learn only a simple decision boundary. A multilayer perceptron (MLP) connects multiple neurons in layers, allowing it to learn more complex patterns.

## 5. Build the Simple Neural Network

```text
2 Input Features
       ↓
8 Neurons — ReLU
       ↓
4 Neurons — ReLU
       ↓
1 Neuron — Sigmoid
       ↓
Binary Output
```

In [ ]:
model = Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(8, activation="relu"),
    Dense(4, activation="relu"),
    Dense(1, activation="sigmoid")
])

## 6. Display the Architecture

The input accepts two values per sample. The first hidden layer has 8 neurons, the second has 4 neurons, and the output layer has 1 neuron. **Trainable parameters** are the weights and biases that the network adjusts while learning. The summary reports how many each layer contains.

In [ ]:
model.summary()

## 7. Activation Functions

### ReLU

```text
ReLU(x) = max(0, x)
```

ReLU is used in both hidden layers. It introduces non-linearity, which helps the network learn complex patterns instead of only straight-line relationships.

### Sigmoid

Sigmoid is used in the output layer for binary prediction. It produces a value between 0 and 1, which can be interpreted as the probability of class 1.

## 8. Compile the Network

- **Binary Cross-Entropy:** measures the difference between the actual binary target and the predicted probability.
- **Adam optimizer:** uses gradients calculated during backpropagation to update weights and biases.
- **Accuracy:** gives a simple indication of whether this demonstration network is learning.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 9. Forward Propagation

```text
Input
  ↓
Hidden Layer 1 → ReLU
  ↓
Hidden Layer 2 → ReLU
  ↓
Output Layer → Sigmoid
  ↓
Prediction
```

During forward propagation: (1) input enters the network, (2) weighted sums are calculated, (3) activation functions are applied, (4) information moves toward the output layer, and (5) the network generates a prediction.

## 10. Loss

```text
Actual Value
      ↘
       Loss
      ↗
Prediction
```

The prediction is compared with the actual target. Binary Cross-Entropy calculates the error, called the loss. A smaller loss generally indicates better predictions.

## 11. Backpropagation Intuition

```text
Prediction
    ↓
Calculate Loss
    ↓
Backpropagation
    ↓
Calculate Gradients
    ↓
Determine how weights contributed to error
```

Backpropagation sends information about the error backward through the network. The **Chain Rule** connects each layer's contribution to the final error. TensorFlow automatically performs differentiation and backpropagation during training, so we do not need to calculate these gradients manually.

## 12. Optimizer and Weight Update

```text
Gradients
    ↓
Adam Optimizer
    ↓
Update Weights & Biases
    ↓
Try Again
```

```text
new weight = old weight - learning_rate × gradient
```

The optimizer uses gradients to adjust the network's parameters, trying to move them toward lower loss. Adam manages the size and direction of these updates automatically.

## 13. Complete Neural Network Learning Cycle

```text
INPUT
  ↓
FORWARD PROPAGATION
  ↓
PREDICTION
  ↓
LOSS FUNCTION
  ↓
BACKPROPAGATION
  ↓
GRADIENTS
  ↓
OPTIMIZER
  ↓
UPDATE WEIGHTS & BIASES
  ↓
REPEAT
```

Each repetition gives the network another opportunity to reduce its error.

## 14. Train the Network

An **epoch** is one complete pass through the training data. The validation split sets aside 20% of the training data to check learning on examples that are not used for weight updates.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

## 15. Visualize Learning

The curves below show how training attempts to reduce Binary Cross-Entropy loss over the epochs.

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs, history.history["loss"], label="Training Loss")
plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
plt.title("Training Loss vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Simple Prediction Demonstration

For eight test samples, a probability of 0.5 or greater becomes class 1; otherwise it becomes class 0.

In [ ]:
sample_count = 8
predicted_probabilities = model.predict(X_test[:sample_count], verbose=0).ravel()
predicted_classes = (predicted_probabilities >= 0.5).astype(int)

print(f"{'Actual Value':<15}{'Predicted Probability':<25}{'Predicted Class'}")
for actual, probability, predicted_class in zip(
    y_test[:sample_count], predicted_probabilities, predicted_classes
):
    print(f"{int(actual):<15}{probability:<25.4f}{predicted_class}")

## Simple Neural Network Architecture

```text
Input Features
      ↓
Dense Hidden Layer
      ↓
ReLU
      ↓
Dense Hidden Layer
      ↓
ReLU
      ↓
Output Layer
      ↓
Sigmoid
      ↓
Prediction
```

The Week 6 concepts connect in one learning process:

```text
Perceptron → Layers → Activation Functions → Forward Propagation
→ Prediction → Loss Function → Backpropagation → Gradients
→ Optimizer → Weight Update
```

## Conclusion

A perceptron is the basic building block of a neural network, and multiple neurons form layers. ReLU introduces non-linearity in the hidden layers, while Sigmoid produces a binary output probability. Forward propagation generates predictions, and Binary Cross-Entropy measures their error. Backpropagation calculates gradients, which Adam uses to update the weights and biases. Repeating this cycle allows the neural network to learn.